# NCAA Basketball Tournament — Feature Engineering

## March Machine Learning Mania 2026

In this notebook we build the features that will power our tournament prediction model.

**What we're predicting:** For every possible matchup in the 2026 NCAA tournament, predict the probability that the lower-seeded team (by TeamID, not seed number) wins.

**Why Brier Score?** The competition uses Brier Score = mean((p - y)²), not accuracy. This rewards *calibrated* probabilities — saying 0.7 when the true probability is 0.7 is better than saying 1.0. Extreme overconfidence is penalised heavily.

**Our approach:** Engineer per-team, per-season features → represent each matchup as the *difference* between two teams' features → train a classifier to predict win probability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── Paths ──────────────────────────────────────────────────────────────────
DATA = Path("/kaggle/input/march-machine-learning-mania-2026")

# Verify data is present
csv_files = list(DATA.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
print("Key files:", [f.name for f in csv_files if "Compact" in f.name or "Massey" in f.name][:6])

## 1. Data Tour

The competition provides 30+ CSV files. We use four main sources:

| File | Content | Rows |
|------|---------|------|
| `MRegularSeasonCompactResults` | Men's game-by-game results 1985–2025 | ~180k |
| `MRegularSeasonDetailedResults` | Men's box scores 2003–2025 | ~120k |
| `MMasseyOrdinals` | 196 external rating systems (Pomeroy, Sagarin, etc.) | ~5.7M |
| `MNCAATourneySeeds` | Tournament seeds 1985–2025 | ~8k |

Each game row has `WTeamID` (winner), `LTeamID` (loser), `WScore`, `LScore`, `DayNum` (day of season), and `WLoc` (H/A/N).

In [ ]:
# Quick look at each key file
compact  = pd.read_csv(DATA / "MRegularSeasonCompactResults.csv")
detailed = pd.read_csv(DATA / "MRegularSeasonDetailedResults.csv")
massey   = pd.read_csv(DATA / "MMasseyOrdinals.csv")
seeds    = pd.read_csv(DATA / "MNCAATourneySeeds.csv")

print("=== Regular Season Compact ===")
print(compact.shape, "\n", compact.head(3))

print("\n=== Regular Season Detailed ===")
print(detailed.shape, "\n", detailed.columns.tolist())

print("\n=== Massey Ordinals ===")
print(massey.shape)
print("Systems:", massey["SystemName"].nunique(), "unique rating systems")
print(massey.head(3))

print("\n=== Tournament Seeds ===")
print(seeds.shape, "\n", seeds.head(3))

## 2. Efficiency Metrics (Four-Factor Model)

Raw points and win% don't tell the full story. A team scoring 80 points in 60 possessions is very different from one scoring 80 in 75 possessions.

We compute **adjusted efficiency** using the Kubatko possession formula:

> Possessions ≈ FGA − ORB + TO + 0.44 × FTA

Then:
- **Offensive efficiency (off_eff)** = 100 × points scored / possessions
- **Defensive efficiency (def_eff)** = 100 × points allowed / possessions
- **Net efficiency (net_eff)** = off_eff − def_eff ← our primary quality metric

We also compute the **Four Factors** (Dean Oliver):
- **eFG%** = (FGM + 0.5 × FGM3) / FGA — shooting quality
- **OREB%** = OR / (OR + opponent DR) — offensive rebounding
- **TOV%** = TO / possessions — turnover rate
- **FT Rate** = FTA / FGA — free throw generation

These 4 factors explain ~95% of scoring variance at the team level.

In [ ]:
def build_efficiency_features(compact_df, detailed_df):
    """Aggregate box-score stats into per-(Season, TeamID) efficiency metrics."""
    det = detailed_df.copy()

    # Reshape: winner perspective + loser perspective
    win_rows = pd.DataFrame({
        "Season": det.Season, "TeamID": det.WTeamID,
        "own_pts": det.WScore, "opp_pts": det.LScore,
        "FGM": det.WFGM, "FGA": det.WFGA, "FGM3": det.WFGM3,
        "FTA": det.WFTA, "OR": det.WOR, "DR": det.WDR, "TO": det.WTO,
        "oFGA": det.LFGA, "oOR": det.LOR, "oTO": det.LTO, "oFTA": det.LFTA, "oDR": det.LDR,
        "won": 1,
    })
    loss_rows = pd.DataFrame({
        "Season": det.Season, "TeamID": det.LTeamID,
        "own_pts": det.LScore, "opp_pts": det.WScore,
        "FGM": det.LFGM, "FGA": det.LFGA, "FGM3": det.LFGM3,
        "FTA": det.LFTA, "OR": det.LOR, "DR": det.LDR, "TO": det.LTO,
        "oFGA": det.WFGA, "oOR": det.WOR, "oTO": det.WTO, "oFTA": det.WFTA, "oDR": det.WDR,
        "won": 0,
    })
    games = pd.concat([win_rows, loss_rows], ignore_index=True)
    games["margin"] = games["own_pts"] - games["opp_pts"]

    grp  = games.groupby(["Season", "TeamID"])
    sums = grp[["own_pts","opp_pts","FGA","FGM","FGM3","FTA","OR","DR","TO",
                "oFGA","oOR","oTO","oFTA","oDR"]].sum().reset_index()
    base = grp.agg(win_pct=("won","mean"), avg_margin=("margin","mean")).reset_index()
    agg  = base.merge(sums, on=["Season","TeamID"])

    own_poss = (agg.FGA - agg.OR  + agg.TO  + 0.44*agg.FTA).replace(0, np.nan)
    opp_poss = (agg.oFGA - agg.oOR + agg.oTO + 0.44*agg.oFTA).replace(0, np.nan)

    agg["off_eff"]  = agg.own_pts / own_poss * 100
    agg["def_eff"]  = agg.opp_pts / opp_poss * 100
    agg["net_eff"]  = agg.off_eff - agg.def_eff
    agg["efg_pct"]  = (agg.FGM + 0.5*agg.FGM3) / agg.FGA
    agg["oreb_pct"] = agg.OR  / (agg.OR  + agg.oDR)
    agg["dreb_pct"] = agg.DR  / (agg.DR  + agg.oOR)
    agg["to_pct"]   = agg.TO  / own_poss
    agg["ft_rate"]  = agg.FTA / agg.FGA

    return agg[["Season","TeamID","win_pct","avg_margin","net_eff","off_eff","def_eff",
                "efg_pct","oreb_pct","dreb_pct","to_pct","ft_rate"]].copy()

eff_m = build_efficiency_features(compact, detailed)
print(f"Efficiency features shape: {eff_m.shape}")
print(eff_m.describe().round(2))

## 3. Elo Ratings with Margin-of-Victory

Efficiency metrics summarise the *average* season. But Elo captures *trajectory* — it updates after every game, so it knows whether a team is peaking or fading.

**Standard Elo update:**
```
expected_A = 1 / (1 + 10^((elo_B - elo_A) / 400))
delta = K × (outcome − expected_A)
```

**Margin-of-victory scaling (key improvement):**
A 30-point win should count more than a 1-point win. We scale the K-factor:
```
K_eff = 20 × (1 + margin/20)^0.6
```
This is the formula used by 4th-place winners in this competition.

**Season carryover (75/25 mean reversion):**
Program quality persists across seasons — Duke is reliably strong year after year. We carry forward 75% of each team's prior Elo:
```
elo_start = prev_season_elo × 0.75 + 1500 × 0.25
```
New teams initialise at 1500 (the global mean).

In [ ]:
def build_elo_features(compact_df):
    """Margin-of-victory Elo with season carryover (75/25 mean reversion)."""
    seasons = sorted(compact_df["Season"].unique())
    elo    = {}   # {TeamID: float}
    records = []

    for season in seasons:
        season_games = compact_df[compact_df["Season"] == season].sort_values("DayNum")
        teams = set(season_games["WTeamID"]) | set(season_games["LTeamID"])

        # Carryover / initialise
        for tid in teams:
            elo[tid] = elo[tid] * 0.75 + 1500 * 0.25 if tid in elo else 1500.0

        k_wins = {tid: 0.0 for tid in teams}

        for row in season_games.itertuples(index=False):
            w, l   = int(row.WTeamID), int(row.LTeamID)
            margin = max(float(row.WScore - row.LScore), 1.0)
            ew     = elo.get(w, 1500.0)
            el     = elo.get(l, 1500.0)
            exp_w  = 1 / (1 + 10 ** ((el - ew) / 400))
            k_eff  = 20 * (1 + margin / 20) ** 0.6
            delta  = k_eff * (1 - exp_w)
            elo[w] += delta
            elo[l] -= delta
            k_wins[w] += k_eff

        for tid in teams:
            records.append({
                "Season": season, "TeamID": tid,
                "elo_rating": elo[tid],
                "elo_k_weighted_wins": k_wins[tid],
            })

    return pd.DataFrame(records)

elo_m = build_elo_features(compact)
print(f"Elo features shape: {elo_m.shape}")

# Top 10 teams by Elo in most recent season
latest = elo_m[elo_m["Season"] == elo_m["Season"].max()].sort_values("elo_rating", ascending=False)
print("\nTop 10 teams by Elo (most recent season):")
print(latest.head(10)[["TeamID","elo_rating"]].to_string(index=False))

## 4. Massey Ordinals — 196 Rating Systems → 2 PCA Dimensions

The competition provides `MMasseyOrdinals.csv` — rankings from 196 external systems (Ken Pomeroy, Sagarin, Moore, RPI, BPI, and 191 others). Each system has different methodology and blind spots.

**Problem:** Picking 3 systems and averaging them leaves signal on the table. But using all 196 as separate features causes severe multicollinearity — many systems rank teams nearly identically.

**Solution: PCA (Principal Component Analysis)**

PCA finds orthogonal axes that explain maximum variance across all systems:
- **PC1** (~80-97% variance): the "consensus ranking" — roughly, the average of all systems
- **PC2** (~1-10% variance): where systems *disagree* — this captures teams that are ranked very differently by, say, pace-adjusted vs. raw-points systems

We keep only systems with ≥50% team coverage to avoid noise from niche systems with sparse data.

In [ ]:
def build_massey_pca_features(massey_df):
    """PCA across all Massey systems with >= 50% coverage. Returns massey_pc1, massey_pc2."""
    df = massey_df[massey_df["RankingDayNum"] <= 133].copy()

    # Last available rank per (Season, TeamID, System)
    df = df.sort_values("RankingDayNum")
    last = df.groupby(["Season","TeamID","SystemName"])["OrdinalRank"].last().reset_index()

    records = []
    for season in sorted(last["Season"].unique()):
        sdf   = last[last["Season"] == season]
        pivot = sdf.pivot_table(index="TeamID", columns="SystemName",
                                values="OrdinalRank", aggfunc="first")

        # Keep systems covering >= 50% of teams
        coverage = pivot.notna().sum() / len(pivot)
        keep     = coverage[coverage >= 0.5].index.tolist()
        if len(keep) < 2:
            for tid in pivot.index:
                records.append({"Season": season, "TeamID": int(tid),
                                "massey_pc1": np.nan, "massey_pc2": np.nan})
            continue

        pivot = pivot[keep]
        # Flip sign: lower rank number = better → higher score = better
        pivot = pivot.max().max() - pivot + 1

        # Fill NaN with column mean, then standardise
        pivot_filled = pivot.fillna(pivot.mean())
        mat = StandardScaler().fit_transform(pivot_filled)

        n_comp = min(2, mat.shape[1])
        pca    = PCA(n_components=n_comp, random_state=42)
        pcs    = pca.fit_transform(mat)

        # Align PC1: top-quartile teams should have positive PC1
        row_means = pivot.mean(axis=1).values
        top_q_idx = np.where(row_means > np.percentile(row_means, 75))[0]
        if len(top_q_idx) > 0 and pcs[top_q_idx, 0].mean() < 0:
            pcs[:, 0] = -pcs[:, 0]

        for i, tid in enumerate(pivot.index):
            records.append({
                "Season": season, "TeamID": int(tid),
                "massey_pc1": float(pcs[i, 0]),
                "massey_pc2": float(pcs[i, 1]) if n_comp >= 2 else np.nan,
            })

    result = pd.DataFrame(records)
    if len(last["Season"].unique()) > 0:
        last_season = sorted(last["Season"].unique())[-1]
        s = last[last["Season"] == last_season]
        p = s.pivot_table(index="TeamID", columns="SystemName", values="OrdinalRank", aggfunc="first")
        cov = p.notna().sum() / len(p)
        n_kept = (cov >= 0.5).sum()
        print(f"Last season: {last_season}, {n_kept} systems kept (from {massey_df['SystemName'].nunique()} total)")
    print(f"Massey PCA features: {result.shape}")
    return result

massey_pca_m = build_massey_pca_features(massey)
print(massey_pca_m.describe().round(3))

## 5. Late-Season Momentum

A team's full-season average masks whether they are *peaking* or *fading* entering the tournament.

Consider two teams with identical 20-10 records:
- Team A: went 8-2 in their last 10 games (hot)
- Team B: went 2-8 in their last 10 games (cold)

These teams have the same seed and same win%, but very different tournament prospects.

We compute three momentum features from the **last 10 regular season games** (by `DayNum`, excluding conference tournament games):
- `recent_win_pct` — win rate over last 10 games
- `recent_net_margin` — average point margin over last 10 games
- `streak` — current streak entering the tournament (positive = wins, negative = losses)

In [ ]:
def build_momentum_features(compact_df, n_games=10):
    """Last-N-games momentum features per (Season, TeamID). Excludes conf tourney games (DayNum >= 132)."""
    comp = compact_df[compact_df["DayNum"] < 132].copy()

    win_rows  = comp[["Season","DayNum","WTeamID","WScore","LScore"]].copy()
    win_rows["TeamID"] = win_rows["WTeamID"]
    win_rows["won"]    = 1
    win_rows["margin"] = win_rows["WScore"] - win_rows["LScore"]

    loss_rows  = comp[["Season","DayNum","LTeamID","WScore","LScore"]].copy()
    loss_rows["TeamID"] = loss_rows["LTeamID"]
    loss_rows["won"]    = 0
    loss_rows["margin"] = loss_rows["LScore"] - loss_rows["WScore"]

    games = pd.concat([
        win_rows[["Season","DayNum","TeamID","won","margin"]],
        loss_rows[["Season","DayNum","TeamID","won","margin"]],
    ], ignore_index=True).sort_values(["Season","TeamID","DayNum"])

    last_n = games.groupby(["Season","TeamID"]).tail(n_games)

    agg = last_n.groupby(["Season","TeamID"]).agg(
        recent_win_pct=("won",    "mean"),
        recent_net_margin=("margin", "mean"),
    ).reset_index()

    # Streak: consecutive outcomes from the last game backwards (uses all season games)
    def compute_streak(grp):
        outcomes = grp.sort_values("DayNum")["won"].values
        if len(outcomes) == 0:
            return 0
        val, streak = outcomes[-1], 0
        for o in reversed(outcomes):
            if o == val:
                streak += 1
            else:
                break
        return streak if val == 1 else -streak

    streak = (games.groupby(["Season","TeamID"])
              .apply(compute_streak, include_groups=False)
              .reset_index(name="streak"))

    result = agg.merge(streak, on=["Season","TeamID"], how="left")
    result["streak"] = result["streak"].fillna(0).astype(int)
    return result

momentum_m = build_momentum_features(compact)
print(f"Momentum features shape: {momentum_m.shape}")
print(momentum_m.describe().round(3))

## 6. Merging All Features + Correlation Analysis

We merge all feature groups on `(Season, TeamID)` and inspect the correlation matrix. This guides feature selection — highly correlated features (r > 0.85) add redundancy without signal.

Key relationships to verify:
- `elo_rating` should correlate strongly with `net_eff` (both measure team quality) — but not perfectly
- `massey_pc1` should correlate strongly with both (consensus of 196 external systems)
- Momentum features should show lower correlation (they capture recent form, not season average)

In [ ]:
# Merge all features
seeds_df = pd.read_csv(DATA / "MNCAATourneySeeds.csv")
seeds_df["SeedNum"] = seeds_df["Seed"].str[1:3].astype(int)

team_features = (eff_m
    .merge(elo_m,         on=["Season","TeamID"], how="left")
    .merge(massey_pca_m,  on=["Season","TeamID"], how="left")
    .merge(momentum_m,    on=["Season","TeamID"], how="left")
    .merge(seeds_df[["Season","TeamID","SeedNum"]], on=["Season","TeamID"], how="left")
)

print(f"Final feature matrix: {team_features.shape}")
nan_counts = team_features.isnull().sum()
print(f"NaN counts:\n{nan_counts[nan_counts > 0]}")

# Correlation heatmap — tourney teams only (have SeedNum)
tourney_teams = team_features.dropna(subset=["SeedNum"])
feature_cols  = ["SeedNum","net_eff","elo_rating","massey_pc1","massey_pc2",
                 "efg_pct","oreb_pct","to_pct","win_pct","avg_margin",
                 "recent_win_pct","recent_net_margin","streak"]
corr = tourney_teams[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, square=True, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix (tournament teams only)", pad=12)
plt.tight_layout()
plt.show()

## 7. Saving Features

We save the merged feature table as a Parquet file. In the modelling notebook we load this directly and build matchup rows by computing `team1_feature - team2_feature` for each possible pair.

One important note: **we keep features as season-level aggregates** (not matchup-level). The matchup diff is computed at training/prediction time. This means we only need to engineer features once per team per season, not once per matchup (which would be hundreds of thousands of rows).

In [ ]:
out_path = Path("/kaggle/working/team_features_M.parquet")
team_features.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")
print(f"Shape: {team_features.shape}")
print(f"Seasons: {team_features['Season'].min()} – {team_features['Season'].max()}")
print(f"Unique teams: {team_features['TeamID'].nunique()}")
print("\nFeature columns:")
for col in team_features.columns:
    nn = team_features[col].notna().sum()
    print(f"  {col:<30} {nn:>6} non-null ({100*nn/len(team_features):.0f}%)")